# Explore

Scratch space. Anything that turns out to be worth keeping graduates to a
script under `scripts/`; this notebook is not the place where the pipeline
eventually lives.


In [ ]:
from pathlib import Path

import pandas as pd

# The notebook runs from notebooks/, so paths are resolved against the repo
# root instead of the working directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RACE_RESULTS = REPO_ROOT / "data" / "raw" / "race_results"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_parquet(RACE_RESULTS)
df.shape

In [ ]:
df.head()

## Label

No non-finisher is ever classified inside the top 10 in this data, so
`position <= 10` needs no separate handling for retirements.


In [ ]:
df['top10'] = (df.position <= 10).astype(int)
df['top10'].mean()

## First lagged feature

Share of the driver's own previous 5 races that ended in the top 10.

Both `shift` and `rolling` run inside the groupby. Chaining `.rolling()` onto
the result of `groupby(...).shift(1)` instead would run the window over the
whole frame, blind to where one driver ends and the next begins. That happens
to give the right numbers while the NaN that `shift` leaves at each driver's
first race sits inside the window, and stops doing so the moment anything
shortens the window, `min_periods=1` above all.

**Open: this window crosses seasons.** A driver's first race of 2019 is scored
from their last five of 2018, which may be a different team and a different
car. Not yet decided.


In [ ]:
df = df.sort_values(['driverId', 'season', 'round'])

df['top10_rate_last5'] = df.groupby('driverId')['top10'].transform(
    lambda s: s.shift(1).rolling(5).mean()
)

In [ ]:
df[df.driverId == 'leclerc'].sort_values(['season', 'round'])[
    ['season', 'round', 'position', 'top10', 'top10_rate_last5']
].head(8)

## Does the current race leak in?

Counting NaNs shows the window has warmed up, but says nothing about whether
the numbers that are not NaN are the right numbers. The second cell answers
that directly, by rebuilding the feature from scratch in plain Python and
comparing.

`aitken` and `pietro_fittipaldi` report fewer than 5 because they started
fewer than 5 races in total, so `head(5)` has nothing to return.


In [ ]:
df.groupby('driverId')['top10_rate_last5'].apply(lambda x: x.head(5).isna().sum())

In [ ]:
expected = []
for _, g in df.groupby('driverId', sort=False):
    t = g['top10'].tolist()
    for i in range(len(t)):
        expected.append(sum(t[i - 5:i]) / 5 if i >= 5 else float('nan'))

df['top10_rate_last5'].round(10).equals(pd.Series(expected, index=df.index).round(10))

## Split

Time split: everything through 2024 trains, 2025 onward tests (721 rows,
19.5%). Scores are always read pooled and per season, because 2026 is the
first year of the new regulations and a pooled score would average that
boundary away. Why the cut sits at 2025 is in `docs/decisions.md`
(2026-08-17).


In [ ]:
train = df[df.season <= 2024]
test = df[df.season >= 2025]
train.shape, test.shape

## Baseline

One rule, nothing fitted: predict a top-ten finish exactly when the car
starts from `grid <= 10`. Every feature and model from here on has to beat
this score to earn its keep.


In [ ]:
pred = test.grid <= 10
(test['top10'] == pred).mean()

In [ ]:
(test['top10'] == pred).groupby(test.season).mean()

## Fill value for the feature's NaN

Decided 2026-08-17 (`docs/decisions.md`): NaN rows are never dropped, they
are filled with the base rate measured on train only, so nothing the model
consumes is computed from test-period outcomes.

Three measurements below: the fill constant itself; the same rate per
season, which is where the 2026 rate of 10/22 surfaced (the grid grew to
22 cars when the eleventh team joined, so the "half the field" intuition
quietly stopped holding); and how many test rows the fill touches at all
(26 of 721, which caps what the choice of fill value can move the score
by at about 3.6 points).

In [ ]:
(train['top10'] == 1).mean()

In [ ]:
df.groupby('season')['top10'].mean()

In [ ]:
test['top10_rate_last5'].isna().sum()